<a href="https://colab.research.google.com/github/WesalFadhl/Murshid-Wesal-Mashael/blob/main/homework/Homework4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Day 4 Homework — Real Model Evaluation

**Program:** LLM Application Engineering (SDA-AIE-213) — SDAIA

**Developed by:** Wesal Al-Numani & Mashael Al-Mufleh

## Objective

Evaluate the performance of a real local language model using three synthetic public-service FAQ questions. The evaluation includes token counts, inference time, keyword indicators, and manual assessment of answer correctness.

## Evaluation Approach

- Local model inference using Hugging Face Transformers.
- Evaluation of three predefined test cases.
- Measurement of input and output token counts.
- Measurement of inference time.
- Keyword-based and manual semantic evaluation.
- No external inference API calls or paid services.



In [1]:
!pip -q install transformers torch sentencepiece

In [2]:

# Day 4 Homework — Cell 2: Load the Real Model

import time
import json
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODEL_NAME = "google/flan-t5-small"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

model.eval()

print("Model loaded successfully!")
print("Model:", MODEL_NAME)
print("Device:", next(model.parameters()).device)


config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

... (2 outputs left out)

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Model loaded successfully!
Model: google/flan-t5-small
Device: cpu


In [3]:

# Day 4 Homework — Cell 3: Real Model Evaluation

from pathlib import Path

torch.set_num_threads(2)

context = "You can check your application status using its reference number."

questions = [
    "How can I check my application status?",
    "What do I need to check my application status?",
    "How can I find out the status of my application?"
]

rows = []

for i, question in enumerate(questions, 1):
    prompt = (
        f"Answer the question using the context.\n"
        f"Context: {context}\n"
        f"Question: {question}\nAnswer:"
    )

    inputs = tokenizer(
        prompt, return_tensors="pt",
        truncation=True, max_length=256
    )

    start = time.perf_counter()

    with torch.inference_mode():
        tokens = model.generate(
            **inputs,
            max_new_tokens=48,
            do_sample=False,
            num_beams=1
        )

    elapsed = time.perf_counter() - start
    answer = tokenizer.decode(
        tokens[0], skip_special_tokens=True
    ).strip()

    assert answer, "Empty generation; inspect runtime."

    keyword_hit = (
        "reference" in answer.lower()
        and "number" in answer.lower()
    )

    rows.append({
        "id": f"E{i:02d}",
        "question": question,
        "answer": answer,
        "reference_keyword_hit": keyword_hit,
        "input_tokens": int(inputs["input_ids"].shape[-1]),
        "output_tokens_including_special": int(tokens.shape[-1]),
        "inference_seconds": round(elapsed, 3)
    })

result = {
    "mode": "real_local_model",
    "model": MODEL_NAME,
    "generation": {
        "do_sample": False,
        "num_beams": 1,
        "max_new_tokens": 48
    },
    "metric": "reference+number keyword proxy; NOT semantic accuracy",
    "keyword_hits": sum(r["reference_keyword_hit"] for r in rows),
    "total": len(rows),
    "external_inference_api_calls": 0,
    "cases": rows,
    "manual_review": "Review every answer for correctness and unsupported facts."
}

output_dir = Path("murshid_live_outputs")
output_dir.mkdir(exist_ok=True)

(output_dir / "evaluation_result.json").write_text(
    json.dumps(result, indent=2),
    encoding="utf-8"
)

print(json.dumps(result, indent=2))


{
  "mode": "real_local_model",
  "model": "google/flan-t5-small",
  "generation": {
    "do_sample": false,
    "num_beams": 1,
    "max_new_tokens": 48
  },
  "metric": "reference+number keyword proxy; NOT semantic accuracy",
  "keyword_hits": 3,
  "total": 3,
  "external_inference_api_calls": 0,
  "cases": [
    {
      "id": "E01",
      "question": "How can I check my application status?",
      "answer": "Use its reference number",
      "reference_keyword_hit": true,
      "input_tokens": 34,
      "output_tokens_including_special": 6,
      "inference_seconds": 0.362
    },
    {
      "id": "E02",
      "question": "What do I need to check my application status?",
      "answer": "Reference number",
      "reference_keyword_hit": true,
      "input_tokens": 36,
      "output_tokens_including_special": 4,
      "inference_seconds": 0.172
    },
    {
      "id": "E03",
      "question": "How can I find out the status of my application?",
      "answer": "Use its reference numbe

## Evaluation Results & Manual Review

The real local model (`google/flan-t5-small`) was evaluated using three synthetic public-service questions (E01–E03).

**Results:**
- Keyword hits: 3/3 (100%)
- External inference API calls: 0
- All three generated answers were consistent with the provided reference context.

### Manual Evaluation

- **E01:** Correct. The model identified the reference number as the method for checking application status.
- **E02:** Correct. The model identified the reference number as the required information.
- **E03:** Correct. The model provided an appropriate response consistent with the reference context.

### Key Findings

The model successfully generated concise, contextually relevant responses across all three test cases. Inference time ranged from 0.172 to 0.362 seconds in this execution.

However, keyword matching is only a proxy metric and does not establish semantic accuracy or general reliability. Further evaluation with more diverse questions would be necessary before real-world deployment.

**Conclusion:** All three keyword checks passed, and manual review found the responses consistent with the supplied context.